# IR Signal Type Checker — Catching Errors Before Silicon

**SC-NeuroCore v3.14** — The first SNN→FPGA compiler with formal signal type checking.

SC-NeuroCore's intermediate representation (IR) distinguishes four
signal types that cannot be freely mixed:

| Type | Representation | Domain |
|------|---------------|--------|
| `BITSTREAM` | Temporal bit sequence | $\{0,1\}^L$, encodes $p$ via density |
| `RATE` | Scalar probability | $[0,1]$ |
| `SPIKE` | Binary event | $\{0,1\}$, single timestep |
| `FIXED` | Q-format integer | $\mathbb{Z}$, fixed-point |

Connecting a `BITSTREAM` output to a `RATE` input without a
decoder/popcount produces wrong results silently. The type checker
catches this before Verilog emission.

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [1]:
from sc_neurocore.compiler.ir_type_checker import (
    SignalType,
    IRNode,
    IREdge,
    IRTypeError,
    check_ir_types,
    types_compatible,
)

print("SC-NeuroCore IR type checker demo")

SC-NeuroCore IR type checker demo


## 1. Signal Type Compatibility Matrix

Not all connections are valid. A bitstream cannot directly feed
a rate-domain node — it needs a decoder (popcount/L).

In [2]:
all_types = [SignalType.BITSTREAM, SignalType.RATE, SignalType.SPIKE, SignalType.FIXED]
names = ["BITSTREAM", "RATE", "SPIKE", "FIXED"]

print(f"{'src \\ dst':>12s}", end="")
for n in names:
    print(f"  {n:>10s}", end="")
print()
print("-" * (12 + 12 * len(names)))

for i, src in enumerate(all_types):
    print(f"{names[i]:>12s}", end="")
    for dst in all_types:
        compat = types_compatible(src, dst)
        symbol = "  ok" if compat else "  XX"
        print(f"{symbol:>10s}", end="")
    print()

   src \ dst   BITSTREAM        RATE       SPIKE       FIXED
------------------------------------------------------------
   BITSTREAM        ok        XX        XX        XX
        RATE        XX        ok        XX        XX
       SPIKE        ok        XX        ok        XX
       FIXED        XX        XX        XX        ok


## 2. Valid IR Graph

A correct SC pipeline: encoder (RATE→BITSTREAM) → AND synapse
(BITSTREAM→BITSTREAM) → popcount (BITSTREAM→RATE) → LIF
(RATE→SPIKE).

In [3]:
valid_nodes = {
    "input": IRNode(name="input", op="source",
                    input_types=[], output_type=SignalType.RATE),
    "encoder": IRNode(name="encoder", op="encoder",
                      input_types=[SignalType.RATE], output_type=SignalType.BITSTREAM),
    "synapse": IRNode(name="synapse", op="and",
                      input_types=[SignalType.BITSTREAM, SignalType.BITSTREAM],
                      output_type=SignalType.BITSTREAM),
    "weight_enc": IRNode(name="weight_enc", op="encoder",
                         input_types=[SignalType.RATE], output_type=SignalType.BITSTREAM),
    "popcount": IRNode(name="popcount", op="popcount",
                       input_types=[SignalType.BITSTREAM], output_type=SignalType.RATE),
    "lif": IRNode(name="lif", op="lif",
                  input_types=[SignalType.RATE], output_type=SignalType.SPIKE),
}

valid_edges = [
    IREdge(src="input", dst="encoder"),
    IREdge(src="encoder", dst="synapse", dst_port=0),
    IREdge(src="weight_enc", dst="synapse", dst_port=1),
    IREdge(src="synapse", dst="popcount"),
    IREdge(src="popcount", dst="lif"),
]

errors = check_ir_types(valid_nodes, valid_edges)
print(f"Type errors in valid graph: {len(errors)}")
assert len(errors) == 0, "valid graph should have no type errors"

Type errors in valid graph: 0


## 3. Invalid Graph: Missing Decoder

If we skip the popcount and connect a BITSTREAM directly to
the LIF (which expects RATE), the type checker flags it.

In [4]:
broken_nodes = {
    "input": IRNode("input", "source", [], SignalType.RATE),
    "encoder": IRNode("encoder", "encoder",
                      [SignalType.RATE], SignalType.BITSTREAM),
    "lif": IRNode("lif", "lif",
                  [SignalType.RATE], SignalType.SPIKE),
}

broken_edges = [
    IREdge(src="input", dst="encoder"),
    IREdge(src="encoder", dst="lif"),  # BITSTREAM → RATE: invalid!
]

errors = check_ir_types(broken_nodes, broken_edges)
print(f"Type errors found: {len(errors)}")
for err in errors:
    print(f"  {err.src_node} ({err.src_type.name}) → {err.dst_node} ({err.dst_type.name})")
    print(f"  Message: {err.message}")

Type errors found: 1
  encoder (BITSTREAM) → lif (RATE)
  Message: Type mismatch: encoder outputs BITSTREAM but lif port 0 expects RATE. Insert a converter (encoder/decoder).


## 4. Invalid Graph: Missing Encoder

Feeding a RATE scalar directly into an AND synapse
(which expects BITSTREAM) is another common mistake.

In [5]:
broken2_nodes = {
    "input": IRNode("input", "source", [], SignalType.RATE),
    "synapse": IRNode("synapse", "and",
                      [SignalType.BITSTREAM, SignalType.BITSTREAM],
                      SignalType.BITSTREAM),
}

broken2_edges = [
    IREdge(src="input", dst="synapse", dst_port=0),  # RATE → BITSTREAM: invalid!
]

errors2 = check_ir_types(broken2_nodes, broken2_edges)
print(f"Type errors: {len(errors2)}")
for err in errors2:
    print(f"  {err.src_node} ({err.src_type.name}) → {err.dst_node} port {err.message}")

Type errors: 1
  input (RATE) → synapse port Type mismatch: input outputs RATE but synapse port 0 expects BITSTREAM. Insert a converter (encoder/decoder).


## 5. SPIKE → BITSTREAM: Valid Embedding

A single spike embeds naturally into a bitstream (single 1-bit
in a stream of zeros). This is the one cross-domain connection
that the type checker permits without an explicit converter.

In [6]:
embed_nodes = {
    "lif": IRNode("lif", "lif",
                  [SignalType.RATE], SignalType.SPIKE),
    "and_gate": IRNode("and_gate", "and",
                       [SignalType.BITSTREAM, SignalType.BITSTREAM],
                       SignalType.BITSTREAM),
}

embed_edges = [
    IREdge(src="lif", dst="and_gate", dst_port=0),  # SPIKE → BITSTREAM: valid
]

errors3 = check_ir_types(embed_nodes, embed_edges)
print(f"SPIKE → BITSTREAM errors: {len(errors3)} (expected 0)")
print(f"Compatible: {types_compatible(SignalType.SPIKE, SignalType.BITSTREAM)}")

SPIKE → BITSTREAM errors: 0 (expected 0)
Compatible: True


## Summary

| Error type | What happens without checker | How it manifests |
|-----------|----------------------------|------------------|
| BITSTREAM → RATE | LIF reads raw bit pattern as voltage | Wild spiking or silence |
| RATE → BITSTREAM | AND gate multiplies scalar by scalar | Always 0 or 1 |
| FIXED → BITSTREAM | Packed integer misread as temporal bits | Garbled output |

The IR type checker runs before Verilog emission. It is cheap
(single graph traversal, O(E) for E edges) and catches silent
wrong-answer bugs that would otherwise require FPGA bring-up
debugging.

No other SNN-to-FPGA compiler has a formal signal type system.
Brian2/snnTorch/Norse do not emit hardware; Lava's Loihi compiler
uses fixed neuron primitives without cross-domain type checking.